In [2]:
%load_ext autoreload
%autoreload 2

import os
import urllib.parse

import pandas as pd
from tqdm import tqdm
from url_graph import URLGraph

PATH = os.path.dirname(os.path.abspath(__name__))

In [3]:
def undersample_dataset(dataframe: pd.DataFrame, n_rows=None, random_state=42) -> pd.DataFrame:
    # Find the minority class and its count
    counts = dataframe["label"].value_counts()
    minority_class = counts.idxmin()
    minority_class_count = counts.min()

    # Split the df into minority and majority classes
    df_minor = dataframe[dataframe["label"] == minority_class]
    df_major = dataframe[dataframe["label"] != minority_class]

    # Randomly sample the majority class to match the minority class count
    df_major = df_major.sample(minority_class_count, random_state=random_state)

    # Concatenate the undersampled majority class with the minority class
    result = pd.concat([df_major, df_minor], ignore_index=True)
    
    # Shuffle the result to mix the classes
    result = result.sample(frac=1, random_state=random_state).reset_index(drop=True)

    # If n_rows is specified, limit the result to the first n_rows
    if n_rows is not None:
        result = result.head(n_rows)

    return result

In [4]:
def create_graph_from_dataframe(df: pd.DataFrame) -> URLGraph:
    g = URLGraph()

    for _, row in tqdm(df.iterrows(), total=df.shape[0], desc="Processing URLs", unit="url", ncols=100):
        raw_url = str(row["url"]).strip()
        label = str(row["label"]).strip()

        # Parse the URL and add it to the graph.
        try:
            parsed_url = urllib.parse.urlparse(raw_url)
            g.add_url(parsed_url, label == "bad")
        except ValueError as e:
            print(f"Error parsing URL '{raw_url}': {e}")

    return g

In [8]:
DATASET_PATH = os.path.join(PATH, "datasets/data.csv")
N_ROWS = 10_000

df = pd.read_csv(DATASET_PATH, header=0)
df = undersample_dataset(df, n_rows=N_ROWS)
graph = create_graph_from_dataframe(df)

output_path = os.path.join(
    PATH,
    f"graphs/graph_{graph.number_of_nodes()}_nodes.gexf"
)

graph.save(output_path)
print(f"Graph saved to {output_path}")

[nltk_data] Downloading package stopwords to /home/kit-
[nltk_data]     matthewson/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
Processing URLs: 100%|█████████████████████████████████████| 10000/10000 [00:01<00:00, 7256.61url/s]


Graph saved to /home/kit-matthewson/Documents/exeter/masters/coursework/systems/code/graphs/graph_31444_nodes.gexf
